# TF-IDF + LinearSVM Traditional Baseline — 5-Fold CV, Seed 42

This notebook replaces the previous **multi-seed fixed train/validation/test** baseline with the same cross-validation protocol used in the flat model notebook:

- `SEED = 42` only.
- The complete prepared dataset and its fixed five-fold assignments are loaded from `dataset_with_test_fold.xlsx`.
- For each CV round: `test = fold i`, `validation = fold (i + 1) mod 5`, and `train = remaining three folds`.
- TF-IDF vectorizers are fitted on the training folds only.
- Reported test performance is based on out-of-fold predictions across the five held-out test folds.

In [ ]:
# =========================
# 0) Imports and Drive mount
# =========================
import os
import re
import json
import random
import warnings
from pathlib import Path
from typing import Dict, Any, List, Optional, Tuple

import numpy as np
import pandas as pd

from scipy.sparse import hstack

import matplotlib.pyplot as plt

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
)
from sklearn.exceptions import ConvergenceWarning

try:
    from IPython.display import display
except Exception:
    display = print

try:
    from google.colab import drive
    drive.mount("/content/drive/")
except Exception as e:
    print("Google Drive was not mounted automatically. This is normal outside Colab.")
    print("Mount message:", repr(e))

warnings.filterwarnings("ignore", category=ConvergenceWarning)


In [ ]:
# =========================
# 1) Configuration
# =========================
SEED = 42
N_FOLDS = 5
CV_VAL_STRATEGY = "rotate_fold"  # test = fold i, val = fold (i + 1) mod 5

# Fixed dataset created by the dataset-preparation notebook.
# It must already contain the exact zero-based test_fold assignment (0...4).
CV_DATASET_PATH = "data/dataset_with_test_fold.xlsx"

# Output folder.
OUT_DIR = "results/01_char_tfidf_svm"
os.makedirs(OUT_DIR, exist_ok=True)

# If your dataset uses unusual column names, update these.
TEXT_COL = "text"
L1_COL = "L1"
L2_COL = "L2"

# Same as the previous traditional baseline.
SVM_C = 1.0
SVM_CLASS_WEIGHT = "balanced"
SVM_MAX_ITER = 10000

CHAR_NGRAM_RANGE = (3, 5)
WORD_NGRAM_RANGE = (1, 2)
MIN_DF = 2
MAX_DF_WORD = 0.95

print("CV_DATASET_PATH:", CV_DATASET_PATH)
print("OUT_DIR:", OUT_DIR)
print("Seed:", SEED)
print("Folds:", N_FOLDS)
print("Validation strategy:", CV_VAL_STRATEGY)


In [ ]:
# =========================
# 2) Reproducibility and I/O helpers
# =========================
def seed_everything(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)


def clean_text(text: Any) -> str:
    if pd.isna(text):
        return ""
    text = str(text).strip()
    text = re.sub(r"\s+", " ", text)
    return text


def norm_label(x: Any) -> str:
    """
    Normalize labels consistently across all folds.

    This keeps compatibility with your previous baseline style:
    "functional suitability" -> "functionalsuitability"
    "time-behaviour" -> "timebehaviour"
    """
    if pd.isna(x):
        return ""
    x = str(x).strip().lower()
    x = re.sub(r"[\s_\-]+", "", x)
    x = re.sub(r"[^a-z0-9]", "", x)
    return x


def read_table(path: str) -> pd.DataFrame:
    path = str(path)
    if not os.path.exists(path):
        raise FileNotFoundError(path)
    if path.lower().endswith((".xlsx", ".xls")):
        return pd.read_excel(path)
    if path.lower().endswith(".csv"):
        return pd.read_csv(path)
    raise ValueError(f"Unsupported file type: {path}")


def standardize_columns(df: pd.DataFrame, source_split: str) -> pd.DataFrame:
    """
    Convert common column-name variants to: id, text, L1, L2.
    L1 is required because the final report includes HP, HR, HF1, per-parent,
    and hierarchical error decomposition.
    """
    col_map = {}

    for c in df.columns:
        lc = str(c).strip().lower()
        if c == TEXT_COL or lc in ["text", "requirement", "requirement_clean", "requirement text", "req", "sentence"]:
            col_map[c] = "text"
        elif c == L2_COL or lc in ["l2", "level2", "level_2", "subclass", "sub_class", "attribute", "label"]:
            col_map[c] = "L2"
        elif c == L1_COL or lc in ["l1", "level1", "level_1", "parent", "category", "mainclass", "main_class"]:
            col_map[c] = "L1"
        elif lc in ["id", "#", "index", "req_id", "requirement_id"]:
            col_map[c] = "id"

    df = df.rename(columns=col_map).copy()

    required = ["text", "L1", "L2"]
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(
            f"{source_split} is missing required columns {missing}. "
            f"Available columns: {list(df.columns)}. "
            "For the same reporting as the flat CV notebook, L1 and L2 are both needed."
        )

    if "id" not in df.columns:
        df = df.reset_index(drop=True)
        df["id"] = df.index.astype(str)

    keep = ["id", "text", "L1", "L2"]
    df = df[keep].copy()

    df["source_split"] = source_split
    df["orig_id"] = df["id"].astype(str)
    df["id"] = source_split + "__" + df["orig_id"].astype(str)

    df["text"] = df["text"].apply(clean_text)
    df["L1"] = df["L1"].apply(norm_label)
    df["L2"] = df["L2"].apply(norm_label)

    df = df[df["text"].str.len() > 0].copy()
    df = df[df["L1"].str.len() > 0].copy()
    df = df[df["L2"].str.len() > 0].copy()

    return df.reset_index(drop=True)


def load_table(path: str, source_split: str) -> pd.DataFrame:
    raw = read_table(path)
    return standardize_columns(raw, source_split)


def load_full_dataset_with_saved_folds(path: str, n_folds: int = 5) -> pd.DataFrame:
    """Load and validate the already-prepared dataset and fixed fold assignments."""
    df = read_table(path).copy()
    required = ["id", "text", "L1", "L2", "cv_index", "test_fold"]
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(
            f"Saved CV dataset is missing required columns: {missing}. "
            "Use full_dataset_with_test_fold.xlsx produced by the preparation notebook."
        )

    if df[required].isna().any().any():
        bad_columns = df[required].columns[df[required].isna().any()].tolist()
        raise ValueError(f"Saved CV dataset contains missing values in: {bad_columns}")

    df["test_fold"] = pd.to_numeric(df["test_fold"], errors="raise").astype(int)
    expected_folds = set(range(n_folds))
    observed_folds = set(df["test_fold"].unique().tolist())
    if observed_folds != expected_folds:
        raise ValueError(
            f"Expected saved folds {sorted(expected_folds)}, but found {sorted(observed_folds)}."
        )

    if df["cv_index"].duplicated().any():
        raise ValueError("cv_index is not unique in the saved CV dataset.")
    if df["id"].astype(str).duplicated().any():
        raise ValueError("id is not unique in the saved CV dataset.")

    # Keep saved values unchanged; do not repeat cleaning or normalization here.
    return df.sort_values("cv_index").reset_index(drop=True)


def build_l2_to_l1_map(df: pd.DataFrame) -> Dict[str, str]:
    conflicts = df.groupby("L2")["L1"].nunique()
    bad = conflicts[conflicts > 1]
    if len(bad) > 0:
        examples = bad.index.tolist()[:10]
        raise ValueError(
            "Some L2 labels map to more than one L1 after normalization. "
            f"Fix label normalization first. Examples: {examples}"
        )
    return df.groupby("L2")["L1"].first().to_dict()


def safe_to_excel(df: pd.DataFrame, path: str, index: bool = False):
    try:
        df.to_excel(path, index=index)
    except Exception as e:
        print(f"Could not write Excel file: {path}")
        print("Reason:", repr(e))


def safe_name(s: str) -> str:
    s = str(s)
    s = re.sub(r"[^a-zA-Z0-9_.-]+", "_", s)
    return s.strip("_")


In [ ]:
# =========================
# 3) Load the prepared dataset and saved fold assignments
# =========================
seed_everything(SEED)

full_df = load_full_dataset_with_saved_folds(CV_DATASET_PATH, n_folds=N_FOLDS)
l2_to_l1 = build_l2_to_l1_map(full_df)

ALL_L2_LABELS = sorted(full_df["L2"].unique().tolist())
ALL_L1_LABELS = sorted(full_df["L1"].unique().tolist())

print("Full dataset:", full_df.shape)
print("L1 classes:", len(ALL_L1_LABELS), ALL_L1_LABELS)
print("L2 classes:", len(ALL_L2_LABELS))
print("\nSmallest L2 supports:")
display(full_df["L2"].value_counts().sort_values().head(10).rename("support").to_frame())

rare = full_df["L2"].value_counts()
rare = rare[rare < N_FOLDS]
if len(rare) > 0:
    print("\nWARNING: Some L2 classes have fewer samples than N_FOLDS.")
    print("These classes cannot appear in every test fold; fold-level metrics may be unstable.")
    display(rare.rename("support").to_frame())

print("\nLoaded fixed fold sizes:")
display(full_df["test_fold"].value_counts().sort_index().rename("n").to_frame())


In [ ]:
# =========================
# 4) Reconstruct CV roles from the saved test_fold column
# =========================
def make_rotating_cv_splits_from_saved_folds(
    df: pd.DataFrame,
    n_folds: int = 5,
) -> List[Dict[str, Any]]:
    """Rebuild train/validation/test roles without generating new folds."""
    folds = [
        np.flatnonzero(df["test_fold"].to_numpy() == fold_id).astype(int)
        for fold_id in range(n_folds)
    ]
    if any(len(indices) == 0 for indices in folds):
        raise ValueError("At least one saved test fold is empty.")

    all_idx = set(range(len(df)))
    split_specs = []
    split_rows = []

    for test_fold in range(n_folds):
        val_fold = (test_fold + 1) % n_folds
        test_idx = folds[test_fold]
        val_idx = folds[val_fold]
        train_idx = np.asarray(
            sorted(list(all_idx - set(test_idx.tolist()) - set(val_idx.tolist()))),
            dtype=int
        )

        spec = {
            "fold": int(test_fold),
            "val_fold": int(val_fold),
            "train_idx": train_idx,
            "val_idx": val_idx,
            "test_idx": test_idx,
            "train_size": int(len(train_idx)),
            "val_size": int(len(val_idx)),
            "test_size": int(len(test_idx)),
        }
        split_specs.append(spec)

        for role, idxs in [("train", train_idx), ("val", val_idx), ("test", test_idx)]:
            tmp = df.iloc[idxs].copy()
            counts = tmp.groupby(["L1", "L2"]).size().reset_index(name="n")
            counts["fold"] = test_fold
            counts["role"] = role
            split_rows.append(counts)

        train_labels = set(df.iloc[train_idx]["L2"])
        val_test_labels = set(df.iloc[val_idx]["L2"]) | set(df.iloc[test_idx]["L2"])
        missing_train = sorted(val_test_labels - train_labels)
        if missing_train:
            print(f"WARNING fold {test_fold}: these L2 labels occur in val/test but not train:", missing_train)
            print("Consider fewer folds if this happens.")

    split_audit = pd.concat(split_rows, ignore_index=True)
    split_audit_path = os.path.join(OUT_DIR, "cv_fold_distribution_by_L1_L2.csv")
    split_audit.to_csv(split_audit_path, index=False)
    safe_to_excel(split_audit, os.path.join(OUT_DIR, "cv_fold_distribution_by_L1_L2.xlsx"))
    print("Saved split audit:", split_audit_path)

    return split_specs


cv_splits = make_rotating_cv_splits_from_saved_folds(full_df, n_folds=N_FOLDS)

fold_overview = pd.DataFrame([
    {k: v for k, v in s.items() if not k.endswith("_idx")}
    for s in cv_splits
])
display(fold_overview)


# Optional archival helper retained for reproducibility.
# The experiment does not call it because it now reads the saved dataset directly.
def save_cv_dataset_and_fold_artifacts(
    df: pd.DataFrame,
    split_specs: List[Dict[str, Any]],
    out_dir: str,
) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    fold_data_dir = os.path.join(out_dir, "fold_data")
    os.makedirs(fold_data_dir, exist_ok=True)

    # Each row is assigned to exactly one held-out test fold.
    dataset_with_folds = df.copy()
    dataset_with_folds["test_fold"] = -1
    for spec in split_specs:
        dataset_with_folds.loc[spec["test_idx"], "test_fold"] = int(spec["fold"])
    dataset_with_folds["test_fold"] = dataset_with_folds["test_fold"].astype(int)

    if (dataset_with_folds["test_fold"] < 0).any():
        raise ValueError("At least one row was not assigned to a test fold.")
    if dataset_with_folds["cv_index"].duplicated().any():
        raise ValueError("cv_index must be unique before saving fold assignments.")

    assigned_csv = os.path.join(out_dir, "full_dataset_with_test_fold.csv")
    assigned_xlsx = os.path.join(out_dir, "full_dataset_with_test_fold.xlsx")
    dataset_with_folds.to_csv(assigned_csv, index=False, encoding="utf-8-sig")
    safe_to_excel(dataset_with_folds, assigned_xlsx)

    # Save the train, validation, and test rows for every experimental fold.
    membership_rows = []
    for spec in split_specs:
        fold_id = int(spec["fold"])
        fold_dir = os.path.join(fold_data_dir, f"fold_{fold_id}")
        os.makedirs(fold_dir, exist_ok=True)

        for role, index_key in [("train", "train_idx"), ("val", "val_idx"), ("test", "test_idx")]:
            indices = np.asarray(spec[index_key], dtype=int)
            split_df = df.iloc[indices].copy()
            split_df.insert(0, "experiment_fold", fold_id)
            split_df.insert(1, "role", role)
            split_df.to_csv(
                os.path.join(fold_dir, f"fold_{fold_id}_{role}.csv"),
                index=False,
                encoding="utf-8-sig",
            )

            membership_rows.append(
                split_df[["experiment_fold", "role", "cv_index", "id", "L1", "L2"]]
            )

    fold_membership = pd.concat(membership_rows, ignore_index=True)
    fold_membership.to_csv(
        os.path.join(out_dir, "cv_fold_membership_all_roles.csv"),
        index=False,
        encoding="utf-8-sig",
    )

    # Overall sizes for train/validation/test in each experimental fold.
    role_sizes = (
        fold_membership.groupby(["experiment_fold", "role"])
        .size()
        .reset_index(name="n")
    )
    role_sizes["percentage_of_full_dataset"] = (
        100.0 * role_sizes["n"] / len(df)
    ).round(2)

    # L1 and L2 distributions for each role within every experimental fold.
    l1_distribution = (
        fold_membership.groupby(["experiment_fold", "role", "L1"])
        .size()
        .reset_index(name="n")
    )
    l1_distribution["percentage_within_role"] = (
        100.0 * l1_distribution["n"]
        / l1_distribution.groupby(["experiment_fold", "role"])["n"].transform("sum")
    ).round(2)

    l2_distribution = (
        fold_membership.groupby(["experiment_fold", "role", "L1", "L2"])
        .size()
        .reset_index(name="n")
    )
    l2_distribution["percentage_within_role"] = (
        100.0 * l2_distribution["n"]
        / l2_distribution.groupby(["experiment_fold", "role"])["n"].transform("sum")
    ).round(2)

    distribution_workbook = os.path.join(out_dir, "cv_fold_distribution_summary.xlsx")
    with pd.ExcelWriter(distribution_workbook, engine="openpyxl") as writer:
        fold_overview.to_excel(writer, sheet_name="Fold Overview", index=False)
        role_sizes.to_excel(writer, sheet_name="Role Sizes", index=False)
        l1_distribution.to_excel(writer, sheet_name="L1 Distribution", index=False)
        l2_distribution.to_excel(writer, sheet_name="L2 Distribution", index=False)

    role_sizes.to_csv(os.path.join(out_dir, "cv_fold_role_sizes.csv"), index=False)
    l1_distribution.to_csv(os.path.join(out_dir, "cv_fold_L1_distribution.csv"), index=False)
    l2_distribution.to_csv(os.path.join(out_dir, "cv_fold_L2_distribution.csv"), index=False)

    print("\nSaved exact CV dataset and fold artifacts:")
    print(" -", assigned_csv)
    print(" -", assigned_xlsx)
    print(" -", distribution_workbook)
    print(" - per-fold train/val/test files in:", fold_data_dir)

    return dataset_with_folds, role_sizes, l1_distribution, l2_distribution


# Verify that every row is used once as held-out test data across the five rounds.
all_saved_test_indices = np.concatenate([s["test_idx"] for s in cv_splits])
if len(all_saved_test_indices) != len(full_df):
    raise ValueError("Saved test folds do not cover the complete dataset exactly once.")
if len(np.unique(all_saved_test_indices)) != len(full_df):
    raise ValueError("Saved test folds contain duplicate row assignments.")
print("Validated: saved test folds cover every requirement exactly once.")


In [ ]:
# =========================
# 5) Metrics, reports, and plotting helpers
# =========================
def flat_metrics_from_labels(
    y_true: List[str],
    y_pred: List[str],
    labels: Optional[List[str]] = None
) -> Dict[str, float]:
    return {
        "macro_f1": float(f1_score(y_true, y_pred, labels=labels, average="macro", zero_division=0)),
        "weighted_f1": float(f1_score(y_true, y_pred, labels=labels, average="weighted", zero_division=0)),
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "balanced_acc": float(balanced_accuracy_score(y_true, y_pred)),
        "precision_macro": float(precision_score(y_true, y_pred, labels=labels, average="macro", zero_division=0)),
        "recall_macro": float(recall_score(y_true, y_pred, labels=labels, average="macro", zero_division=0)),
    }


def hierarchical_path_metrics(pred_df: pd.DataFrame) -> Dict[str, float]:
    hp_vals, hr_vals, hf_vals = [], [], []

    for r in pred_df.itertuples(index=False):
        true_path = {r.true_L1, r.true_L2}
        pred_path = {r.pred_L1, r.pred_L2}
        inter = len(true_path & pred_path)

        hp = inter / max(len(pred_path), 1)
        hr = inter / max(len(true_path), 1)
        hf = 0.0 if (hp + hr) == 0 else (2 * hp * hr) / (hp + hr)

        hp_vals.append(hp)
        hr_vals.append(hr)
        hf_vals.append(hf)

    return {
        "HP": float(np.mean(hp_vals)) if hp_vals else 0.0,
        "HR": float(np.mean(hr_vals)) if hr_vals else 0.0,
        "HF1": float(np.mean(hf_vals)) if hf_vals else 0.0,
        "parent_accuracy": float(accuracy_score(pred_df["true_L1"], pred_df["pred_L1"])) if len(pred_df) else 0.0,
    }


def error_decomposition(pred_df: pd.DataFrame) -> Dict[str, Any]:
    n = len(pred_df)
    correct_l2 = pred_df["true_L2"].eq(pred_df["pred_L2"])
    correct_parent = pred_df["true_L1"].eq(pred_df["pred_L1"])

    wrong_l2_correct_l1 = (~correct_l2) & correct_parent
    wrong_l1 = ~correct_parent

    return {
        "n_samples": int(n),
        "Correct L2": int(correct_l2.sum()),
        "Wrong L2 but correct L1": int(wrong_l2_correct_l1.sum()),
        "Wrong L1": int(wrong_l1.sum()),
        "Correct L2 pct": float(correct_l2.mean()) if n else 0.0,
        "Wrong L2 but correct L1 pct": float(wrong_l2_correct_l1.mean()) if n else 0.0,
        "Wrong L1 pct": float(wrong_l1.mean()) if n else 0.0,
    }


def classification_report_df(y_true: List[str], y_pred: List[str], labels: List[str]) -> pd.DataFrame:
    rep = classification_report(y_true, y_pred, labels=labels, output_dict=True, zero_division=0)
    rows = []

    for lab in labels:
        d = rep.get(lab, {})
        rows.append({
            "label": lab,
            "precision": d.get("precision", np.nan),
            "recall": d.get("recall", np.nan),
            "f1": d.get("f1-score", np.nan),
            "support": d.get("support", 0),
        })

    out = pd.DataFrame(rows)

    # For fold-level per-class mean/std, zero-support classes should not drag the mean.
    zero_support = out["support"].fillna(0).eq(0)
    out.loc[zero_support, ["precision", "recall", "f1"]] = np.nan

    return out


def per_parent_within_l2_metrics(
    pred_df: pd.DataFrame,
    l2_to_l1: Dict[str, str],
    all_l1_labels: List[str]
) -> pd.DataFrame:
    rows = []

    for parent in all_l1_labels:
        sub = pred_df[pred_df["true_L1"].eq(parent)].copy()
        children = sorted([l2 for l2, p in l2_to_l1.items() if p == parent])
        support = len(sub)

        if support == 0:
            rows.append({
                "parent": parent,
                "support": 0,
                "parent_recall_as_parent": np.nan,
                "within_parent_l2_macro_f1": np.nan,
                "within_parent_l2_weighted_f1": np.nan,
                "sibling_error_rate": np.nan,
                "cross_parent_error_rate": np.nan,
            })
            continue

        correct_l2 = sub["true_L2"].eq(sub["pred_L2"])
        correct_parent = sub["true_L1"].eq(sub["pred_L1"])

        rows.append({
            "parent": parent,
            "support": int(support),
            "parent_recall_as_parent": float(correct_parent.mean()),
            "within_parent_l2_macro_f1": float(
                f1_score(sub["true_L2"], sub["pred_L2"], labels=children, average="macro", zero_division=0)
            ),
            "within_parent_l2_weighted_f1": float(
                f1_score(sub["true_L2"], sub["pred_L2"], labels=children, average="weighted", zero_division=0)
            ),
            "sibling_error_rate": float(((~correct_l2) & correct_parent).mean()),
            "cross_parent_error_rate": float((~correct_parent).mean()),
        })

    return pd.DataFrame(rows)


def prefix_metrics(prefix: str, d: Dict[str, Any]) -> Dict[str, Any]:
    return {f"{prefix}_{k}": v for k, v in d.items()}


def plot_confusion(
    cm: np.ndarray,
    labels: List[str],
    title: str,
    out_path: str,
    annotate_threshold: int = 15
):
    fig_size = max(8, min(18, 0.45 * len(labels)))
    plt.figure(figsize=(fig_size, fig_size))
    plt.imshow(cm)
    plt.title(title)
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.xticks(range(len(labels)), labels, rotation=90)
    plt.yticks(range(len(labels)), labels)

    if len(labels) <= annotate_threshold:
        for i in range(cm.shape[0]):
            for j in range(cm.shape[1]):
                if cm[i, j] != 0:
                    plt.text(j, i, str(cm[i, j]), ha="center", va="center")

    plt.tight_layout()
    plt.savefig(out_path, dpi=220, bbox_inches="tight")
    plt.close()


def save_json(obj: Dict[str, Any], path: str):
    with open(path, "w") as f:
        json.dump(obj, f, indent=2)


def mean_std_summary(df: pd.DataFrame, group_cols: List[str], metric_cols: List[str]) -> pd.DataFrame:
    agg_dict = {"n_folds": ("fold", "nunique")}
    for c in metric_cols:
        agg_dict[f"{c}_mean"] = (c, "mean")
        agg_dict[f"{c}_std"] = (c, "std")

    out = df.groupby(group_cols, dropna=False).agg(**agg_dict).reset_index()

    for c in metric_cols:
        out[f"{c}_mean±std"] = out.apply(
            lambda r: f"{r[f'{c}_mean']:.4f} ± {0.0 if pd.isna(r[f'{c}_std']) else r[f'{c}_std']:.4f}",
            axis=1,
        )

    return out


In [ ]:
# =========================
# 6) TF-IDF feature builders
# =========================
def make_feature_packs(X_train: List[str], X_val: List[str], X_test: List[str]) -> Dict[str, Dict[str, Any]]:
    """
    Fit vectorizers on TRAIN only, then transform validation/test.
    This avoids leakage and matches the CV design of the neural flat model.
    """
    char_vectorizer = TfidfVectorizer(
        analyzer="char_wb",
        ngram_range=CHAR_NGRAM_RANGE,
        lowercase=True,
        sublinear_tf=True,
        min_df=MIN_DF,
        norm="l2",
    )

    word_vectorizer = TfidfVectorizer(
        analyzer="word",
        ngram_range=WORD_NGRAM_RANGE,
        lowercase=True,
        strip_accents="unicode",
        sublinear_tf=True,
        min_df=MIN_DF,
        max_df=MAX_DF_WORD,
        norm="l2",
    )

    Xtr_char = char_vectorizer.fit_transform(X_train)
    Xva_char = char_vectorizer.transform(X_val)
    Xte_char = char_vectorizer.transform(X_test)

    Xtr_word = word_vectorizer.fit_transform(X_train)
    Xva_word = word_vectorizer.transform(X_val)
    Xte_word = word_vectorizer.transform(X_test)

    Xtr_word_char = hstack([Xtr_word, Xtr_char]).tocsr()
    Xva_word_char = hstack([Xva_word, Xva_char]).tocsr()
    Xte_word_char = hstack([Xte_word, Xte_char]).tocsr()

    return {
        "char_tfidf_linearsvm_balanced": {
            "X_train": Xtr_char,
            "X_val": Xva_char,
            "X_test": Xte_char,
            "vectorizers": {"char": char_vectorizer},
        },
        "word_char_tfidf_linearsvm_balanced": {
            "X_train": Xtr_word_char,
            "X_val": Xva_word_char,
            "X_test": Xte_word_char,
            "vectorizers": {"word": word_vectorizer, "char": char_vectorizer},
        },
    }


In [ ]:
# =========================
# 7) Run one CV fold
# =========================
def make_prediction_frame(
    df: pd.DataFrame,
    preds: np.ndarray,
    fold: int,
    split_name: str,
    model_name: str,
    l2_to_l1: Dict[str, str],
) -> pd.DataFrame:
    out = df[["id", "orig_id", "source_split", "cv_index", "text", "L1", "L2"]].copy()
    out = out.rename(columns={"L1": "true_L1", "L2": "true_L2"})
    out["fold"] = int(fold)
    out["split"] = split_name
    out["model"] = model_name
    out["pred_L2"] = preds
    out["pred_L1"] = out["pred_L2"].map(l2_to_l1)

    missing_parent = out["pred_L1"].isna().sum()
    if missing_parent:
        raise ValueError(f"{missing_parent} predicted L2 labels have no L1 mapping.")

    return out


def evaluate_and_save_split(
    pred_df: pd.DataFrame,
    split_name: str,
    labels_l2: List[str],
    labels_l1: List[str],
    run_dir: str,
) -> Dict[str, Any]:
    y_true = pred_df["true_L2"].tolist()
    y_pred = pred_df["pred_L2"].tolist()

    flat = flat_metrics_from_labels(y_true, y_pred)
    h = hierarchical_path_metrics(pred_df)

    metrics = {
        **prefix_metrics(split_name, flat),
        **prefix_metrics(split_name, h),
    }

    if split_name == "test":
        metrics.update(prefix_metrics("test_error", error_decomposition(pred_df)))

    # L2 report and confusion matrix.
    l2_report = classification_report_df(y_true, y_pred, labels_l2)
    l2_report = l2_report.rename(columns={"label": "L2"})
    l2_report["L1"] = l2_report["L2"].map(l2_to_l1)
    l2_report = l2_report[["L1", "L2", "support", "precision", "recall", "f1"]]
    l2_report.to_csv(os.path.join(run_dir, f"{split_name}_classification_report_L2.csv"), index=False)

    cm_l2 = confusion_matrix(y_true, y_pred, labels=labels_l2)
    pd.DataFrame(cm_l2, index=labels_l2, columns=labels_l2).to_csv(
        os.path.join(run_dir, f"{split_name}_confusion_matrix_L2.csv")
    )
    plot_confusion(
        cm_l2,
        labels_l2,
        f"{split_name.upper()} L2 confusion",
        os.path.join(run_dir, f"{split_name}_confusion_matrix_L2.png"),
    )

    # Parent report and confusion matrix.
    parent_report = classification_report_df(
        pred_df["true_L1"].tolist(),
        pred_df["pred_L1"].tolist(),
        labels_l1,
    )
    parent_report = parent_report.rename(columns={"label": "parent"})
    parent_report.to_csv(os.path.join(run_dir, f"{split_name}_classification_report_parent.csv"), index=False)

    cm_l1 = confusion_matrix(pred_df["true_L1"], pred_df["pred_L1"], labels=labels_l1)
    pd.DataFrame(cm_l1, index=labels_l1, columns=labels_l1).to_csv(
        os.path.join(run_dir, f"{split_name}_confusion_matrix_parent.csv")
    )
    plot_confusion(
        cm_l1,
        labels_l1,
        f"{split_name.upper()} parent confusion",
        os.path.join(run_dir, f"{split_name}_confusion_matrix_parent.png"),
        annotate_threshold=20,
    )

    save_json(metrics, os.path.join(run_dir, f"{split_name}_metrics.json"))

    return metrics


def run_one_cv_fold(full_df: pd.DataFrame, split_spec: Dict[str, Any], seed: int = 42):
    fold = int(split_spec["fold"])

    train_df = full_df.iloc[split_spec["train_idx"]].reset_index(drop=True)
    val_df   = full_df.iloc[split_spec["val_idx"]].reset_index(drop=True)
    test_df  = full_df.iloc[split_spec["test_idx"]].reset_index(drop=True)

    X_train = train_df["text"].tolist()
    y_train = train_df["L2"].tolist()

    X_val = val_df["text"].tolist()
    X_test = test_df["text"].tolist()

    print("\n" + "#" * 80)
    print(f"FOLD {fold} | val_fold={split_spec['val_fold']} | train={len(train_df)} val={len(val_df)} test={len(test_df)}")
    print("#" * 80)

    seed_everything(seed)

    # Fit TF-IDF only on training fold.
    feature_packs = make_feature_packs(X_train, X_val, X_test)

    rows = []
    val_preds = []
    test_preds = []

    for model_name, pack in feature_packs.items():
        print("\n" + "=" * 70)
        print(f"Training: {model_name} | fold={fold} | seed={seed}")
        print("=" * 70)

        run_id = f"baseline_cv{fold}_{model_name}_seed{seed}"
        run_dir = os.path.join(OUT_DIR, run_id)
        os.makedirs(run_dir, exist_ok=True)

        clf = LinearSVC(
            C=SVM_C,
            class_weight=SVM_CLASS_WEIGHT,
            random_state=seed,
            max_iter=SVM_MAX_ITER,
        )

        clf.fit(pack["X_train"], y_train)

        val_pred = clf.predict(pack["X_val"])
        test_pred = clf.predict(pack["X_test"])

        val_pred_df = make_prediction_frame(
            val_df, val_pred, fold, "val", model_name, l2_to_l1
        )
        test_pred_df = make_prediction_frame(
            test_df, test_pred, fold, "test", model_name, l2_to_l1
        )

        val_pred_df.to_csv(os.path.join(run_dir, "val_predictions.csv"), index=False)
        test_pred_df.to_csv(os.path.join(run_dir, "test_predictions.csv"), index=False)

        val_metrics = evaluate_and_save_split(
            val_pred_df, "val", ALL_L2_LABELS, ALL_L1_LABELS, run_dir
        )
        test_metrics = evaluate_and_save_split(
            test_pred_df, "test", ALL_L2_LABELS, ALL_L1_LABELS, run_dir
        )

        row = {
            "run_id": run_id,
            "stage": "traditional_baseline_5fold_cv",
            "fold": int(fold),
            "val_fold": int(split_spec["val_fold"]),
            "train_size": int(len(train_df)),
            "val_size": int(len(val_df)),
            "test_size": int(len(test_df)),
            "model": model_name,
            "seed": int(seed),
            "svm_C": float(SVM_C),
            "svm_class_weight": SVM_CLASS_WEIGHT,
            "svm_max_iter": int(SVM_MAX_ITER),
            "char_ngram_range": str(CHAR_NGRAM_RANGE),
            "word_ngram_range": str(WORD_NGRAM_RANGE),
            "min_df": int(MIN_DF),
            "max_df_word": float(MAX_DF_WORD),
            **val_metrics,
            **test_metrics,
            "status": "OK",
            "output_dir": run_dir,
        }

        save_json(row, os.path.join(run_dir, "fold_result_row.json"))

        print("VAL :", " | ".join([f"{k.replace('val_', '')}={v:.4f}" for k, v in val_metrics.items() if k.startswith("val_") and isinstance(v, float)]))
        print("TEST:", " | ".join([f"{k.replace('test_', '')}={v:.4f}" for k, v in test_metrics.items() if k.startswith("test_") and isinstance(v, float)]))

        rows.append(row)
        val_preds.append(val_pred_df)
        test_preds.append(test_pred_df)

    return rows, val_preds, test_preds


In [ ]:
# =========================
# 8) Run all five CV folds
# =========================
def run_cv_experiment():
    all_rows = []
    all_val_preds = []
    all_test_preds = []

    for split_spec in cv_splits:
        try:
            rows, val_pred_frames, test_pred_frames = run_one_cv_fold(
                full_df=full_df,
                split_spec=split_spec,
                seed=SEED,
            )
            all_rows.extend(rows)
            all_val_preds.extend(val_pred_frames)
            all_test_preds.extend(test_pred_frames)

        except Exception as e:
            print("FAILED fold:", split_spec["fold"], repr(e))
            all_rows.append({
                "run_id": f"baseline_cv{split_spec['fold']}_FAILED_seed{SEED}",
                "stage": "traditional_baseline_5fold_cv",
                "fold": int(split_spec["fold"]),
                "val_fold": int(split_spec["val_fold"]),
                "seed": int(SEED),
                "status": "FAILED",
                "error": repr(e),
            })

    results_df = pd.DataFrame(all_rows)
    val_oof = pd.concat(all_val_preds, ignore_index=True) if all_val_preds else pd.DataFrame()
    test_oof = pd.concat(all_test_preds, ignore_index=True) if all_test_preds else pd.DataFrame()

    results_df.to_csv(os.path.join(OUT_DIR, "baseline_cv_fold_results.csv"), index=False)
    safe_to_excel(results_df, os.path.join(OUT_DIR, "baseline_cv_fold_results.xlsx"))

    val_oof.to_csv(os.path.join(OUT_DIR, "baseline_cv_val_predictions_all_folds.csv"), index=False)
    test_oof.to_csv(os.path.join(OUT_DIR, "baseline_cv_test_oof_predictions.csv"), index=False)

    print("\nSaved fold results and out-of-fold predictions in:", OUT_DIR)
    return results_df, val_oof, test_oof


results_df, val_oof, test_oof = run_cv_experiment()
display(results_df)


In [ ]:
# =========================
# 9) Aggregate CV report: fold mean ± std and OOF performance
# =========================
def summarize_cv_fold_results(results_df: pd.DataFrame) -> pd.DataFrame:
    df = results_df.copy()
    if "status" in df.columns:
        df = df[df["status"].eq("OK")].copy()

    group_cols = ["model", "seed"]
    metric_cols = [
        c for c in df.columns
        if c.startswith("val_") or c.startswith("test_")
    ]
    metric_cols = [c for c in metric_cols if pd.api.types.is_numeric_dtype(df[c])]

    summary = mean_std_summary(df, group_cols, metric_cols)

    sort_col = "test_macro_f1_mean"
    if sort_col in summary.columns:
        summary = summary.sort_values(sort_col, ascending=False).reset_index(drop=True)

    return summary


def compute_oof_summary(test_oof: pd.DataFrame) -> pd.DataFrame:
    rows = []

    for model_name, sub in test_oof.groupby("model"):
        flat = flat_metrics_from_labels(
            sub["true_L2"].tolist(),
            sub["pred_L2"].tolist(),
            labels=ALL_L2_LABELS,
        )
        h = hierarchical_path_metrics(sub)
        parent = flat_metrics_from_labels(
            sub["true_L1"].tolist(),
            sub["pred_L1"].tolist(),
            labels=ALL_L1_LABELS,
        )
        err = error_decomposition(sub)

        rows.append({
            "model": model_name,
            "seed": SEED,
            **prefix_metrics("oof_test", flat),
            **prefix_metrics("oof_test", h),
            **prefix_metrics("oof_parent", parent),
            **prefix_metrics("oof_error", err),
        })

    out = pd.DataFrame(rows)
    if "oof_test_macro_f1" in out.columns:
        out = out.sort_values("oof_test_macro_f1", ascending=False).reset_index(drop=True)
    return out


cv_summary = summarize_cv_fold_results(results_df)
oof_summary_df = compute_oof_summary(test_oof)

cv_summary.to_csv(os.path.join(OUT_DIR, "baseline_cv_summary_mean_std_over_folds.csv"), index=False)
safe_to_excel(cv_summary, os.path.join(OUT_DIR, "baseline_cv_summary_mean_std_over_folds.xlsx"))

oof_summary_df.to_csv(os.path.join(OUT_DIR, "baseline_cv_oof_overall_summary.csv"), index=False)
safe_to_excel(oof_summary_df, os.path.join(OUT_DIR, "baseline_cv_oof_overall_summary.xlsx"))

print("Fold mean ± std summary:")
display(cv_summary)

print("OOF test summary over all five held-out test folds:")
display(oof_summary_df)


In [ ]:
# =========================
# 10) Aggregate OOF confusion matrices
# =========================
for model_name, sub in test_oof.groupby("model"):
    model_tag = safe_name(model_name)

    cm_l2_oof = confusion_matrix(sub["true_L2"], sub["pred_L2"], labels=ALL_L2_LABELS)
    cm_parent_oof = confusion_matrix(sub["true_L1"], sub["pred_L1"], labels=ALL_L1_LABELS)

    pd.DataFrame(cm_l2_oof, index=ALL_L2_LABELS, columns=ALL_L2_LABELS).to_csv(
        os.path.join(OUT_DIR, f"cm_oof_L2_{model_tag}.csv")
    )
    pd.DataFrame(cm_parent_oof, index=ALL_L1_LABELS, columns=ALL_L1_LABELS).to_csv(
        os.path.join(OUT_DIR, f"cm_oof_parent_{model_tag}.csv")
    )

    plot_confusion(
        cm_l2_oof,
        ALL_L2_LABELS,
        f"OOF TEST L2 confusion — {model_name}",
        os.path.join(OUT_DIR, f"cm_oof_L2_{model_tag}.png"),
    )
    plot_confusion(
        cm_parent_oof,
        ALL_L1_LABELS,
        f"OOF TEST parent confusion — {model_name}",
        os.path.join(OUT_DIR, f"cm_oof_parent_{model_tag}.png"),
        annotate_threshold=20,
    )

print("Saved aggregate OOF confusion matrices in:", OUT_DIR)


In [ ]:
# =========================
# 11) Per-class analysis over folds and OOF test set
# =========================
per_class_rows = []

for (model_name, fold), sub in test_oof.groupby(["model", "fold"]):
    rep = classification_report_df(
        sub["true_L2"].tolist(),
        sub["pred_L2"].tolist(),
        ALL_L2_LABELS,
    )
    rep = rep.rename(columns={"label": "L2"})
    rep["model"] = model_name
    rep["fold"] = int(fold)
    rep["L1"] = rep["L2"].map(l2_to_l1)
    per_class_rows.append(rep)

per_class_by_fold = pd.concat(per_class_rows, ignore_index=True)

per_class_summary = (
    per_class_by_fold
    .groupby(["model", "L1", "L2"], dropna=False)
    .agg(
        n_test_folds_with_support=("support", lambda s: int((s > 0).sum())),
        total_support=("support", "sum"),
        precision_mean=("precision", "mean"),
        precision_std=("precision", "std"),
        recall_mean=("recall", "mean"),
        recall_std=("recall", "std"),
        f1_mean=("f1", "mean"),
        f1_std=("f1", "std"),
    )
    .reset_index()
    .sort_values(["model", "L1", "f1_mean", "total_support"], ascending=[True, True, True, False])
)

per_class_oof_rows = []
for model_name, sub in test_oof.groupby("model"):
    rep = classification_report_df(
        sub["true_L2"].tolist(),
        sub["pred_L2"].tolist(),
        ALL_L2_LABELS,
    )
    rep = rep.rename(columns={"label": "L2"})
    rep["model"] = model_name
    rep["L1"] = rep["L2"].map(l2_to_l1)
    per_class_oof_rows.append(rep)

per_class_oof = pd.concat(per_class_oof_rows, ignore_index=True)
per_class_oof = per_class_oof[["model", "L1", "L2", "support", "precision", "recall", "f1"]]
per_class_oof = per_class_oof.sort_values(["model", "L1", "f1", "support"], ascending=[True, True, True, False])

per_class_by_fold.to_csv(os.path.join(OUT_DIR, "per_class_by_fold.csv"), index=False)
per_class_summary.to_csv(os.path.join(OUT_DIR, "per_class_summary_mean_std_over_folds.csv"), index=False)
per_class_oof.to_csv(os.path.join(OUT_DIR, "per_class_oof_report.csv"), index=False)

safe_to_excel(per_class_summary, os.path.join(OUT_DIR, "per_class_summary_mean_std_over_folds.xlsx"))
safe_to_excel(per_class_oof, os.path.join(OUT_DIR, "per_class_oof_report.xlsx"))

print("Worst OOF L2 classes by F1 for each model:")
for model_name, sub in per_class_oof.groupby("model"):
    print("\nMODEL:", model_name)
    display(sub.sort_values(["f1", "support"], ascending=[True, False]).head(15))


In [ ]:
# =========================
# 12) Per-parent analysis over folds and OOF test set
# =========================
per_parent_rows = []

for (model_name, fold), sub in test_oof.groupby(["model", "fold"]):
    parent_rep = classification_report_df(
        sub["true_L1"].tolist(),
        sub["pred_L1"].tolist(),
        ALL_L1_LABELS,
    )
    parent_rep = parent_rep.rename(columns={
        "label": "parent",
        "precision": "parent_precision",
        "recall": "parent_recall",
        "f1": "parent_f1",
        "support": "parent_support",
    })
    parent_rep["model"] = model_name
    parent_rep["fold"] = int(fold)

    within = per_parent_within_l2_metrics(sub, l2_to_l1, ALL_L1_LABELS)
    within = within.rename(columns={"support": "within_parent_support"})

    merged = parent_rep.merge(within, on="parent", how="left")
    per_parent_rows.append(merged)

per_parent_by_fold = pd.concat(per_parent_rows, ignore_index=True)

summary_cols = [
    "parent_precision",
    "parent_recall",
    "parent_f1",
    "parent_recall_as_parent",
    "within_parent_l2_macro_f1",
    "within_parent_l2_weighted_f1",
    "sibling_error_rate",
    "cross_parent_error_rate",
]

agg = {
    "n_test_folds_with_support": ("parent_support", lambda s: int((s > 0).sum())),
    "total_support": ("parent_support", "sum"),
}
for c in summary_cols:
    agg[f"{c}_mean"] = (c, "mean")
    agg[f"{c}_std"] = (c, "std")

per_parent_summary = (
    per_parent_by_fold
    .groupby(["model", "parent"], dropna=False)
    .agg(**agg)
    .reset_index()
    .sort_values(["model", "within_parent_l2_macro_f1_mean"], ascending=[True, True])
)

per_parent_oof_rows = []

for model_name, sub in test_oof.groupby("model"):
    parent_rep = classification_report_df(
        sub["true_L1"].tolist(),
        sub["pred_L1"].tolist(),
        ALL_L1_LABELS,
    )
    parent_rep = parent_rep.rename(columns={
        "label": "parent",
        "precision": "parent_precision",
        "recall": "parent_recall",
        "f1": "parent_f1",
        "support": "parent_support",
    })
    parent_rep["model"] = model_name

    within = per_parent_within_l2_metrics(sub, l2_to_l1, ALL_L1_LABELS)
    within = within.rename(columns={"support": "within_parent_support"})

    merged = parent_rep.merge(within, on="parent", how="left")
    per_parent_oof_rows.append(merged)

per_parent_oof = pd.concat(per_parent_oof_rows, ignore_index=True)
per_parent_oof = per_parent_oof.sort_values(["model", "within_parent_l2_macro_f1"], ascending=[True, True])

per_parent_by_fold.to_csv(os.path.join(OUT_DIR, "per_parent_by_fold.csv"), index=False)
per_parent_summary.to_csv(os.path.join(OUT_DIR, "per_parent_summary_mean_std_over_folds.csv"), index=False)
per_parent_oof.to_csv(os.path.join(OUT_DIR, "per_parent_oof_report.csv"), index=False)

safe_to_excel(per_parent_summary, os.path.join(OUT_DIR, "per_parent_summary_mean_std_over_folds.xlsx"))
safe_to_excel(per_parent_oof, os.path.join(OUT_DIR, "per_parent_oof_report.xlsx"))

print("OOF per-parent report:")
display(per_parent_oof)


In [ ]:
# =========================
# 13) Error analysis over folds, by parent, and by class
# =========================
error_rows = []

for (model_name, fold), sub in test_oof.groupby(["model", "fold"]):
    row = {"model": model_name, "fold": int(fold)}
    row.update(error_decomposition(sub))
    error_rows.append(row)

error_by_fold = pd.DataFrame(error_rows)

error_metric_cols = [c for c in error_by_fold.columns if c not in ["model", "fold"]]
error_summary = (
    error_by_fold
    .groupby("model")
    .agg(
        n_folds=("fold", "nunique"),
        **{f"{c}_mean": (c, "mean") for c in error_metric_cols},
        **{f"{c}_std": (c, "std") for c in error_metric_cols},
    )
    .reset_index()
)

err_df = test_oof.copy()

correct_l2 = err_df["true_L2"].eq(err_df["pred_L2"])
correct_parent = err_df["true_L1"].eq(err_df["pred_L1"])

err_df["error_type"] = np.select(
    [correct_l2, (~correct_l2) & correct_parent, ~correct_parent],
    ["Correct L2", "Wrong L2 but correct L1", "Wrong L1"],
    default="Unknown",
)

error_by_parent = (
    err_df.groupby(["model", "true_L1", "error_type"])
    .size()
    .reset_index(name="n")
    .merge(
        err_df.groupby(["model", "true_L1"]).size().reset_index(name="parent_support"),
        on=["model", "true_L1"],
    )
)
error_by_parent["pct_within_parent"] = error_by_parent["n"] / error_by_parent["parent_support"]

error_by_class = (
    err_df.groupby(["model", "true_L1", "true_L2", "error_type"])
    .size()
    .reset_index(name="n")
    .merge(
        err_df.groupby(["model", "true_L1", "true_L2"]).size().reset_index(name="class_support"),
        on=["model", "true_L1", "true_L2"],
    )
)
error_by_class["pct_within_class"] = error_by_class["n"] / error_by_class["class_support"]

error_by_fold.to_csv(os.path.join(OUT_DIR, "error_analysis_by_fold.csv"), index=False)
error_summary.to_csv(os.path.join(OUT_DIR, "error_analysis_summary_mean_std.csv"), index=False)
err_df.to_csv(os.path.join(OUT_DIR, "oof_predictions_with_error_type.csv"), index=False)
error_by_parent.to_csv(os.path.join(OUT_DIR, "error_analysis_by_parent.csv"), index=False)
error_by_class.to_csv(os.path.join(OUT_DIR, "error_analysis_by_class.csv"), index=False)

safe_to_excel(error_summary, os.path.join(OUT_DIR, "error_analysis_summary_mean_std.xlsx"))
safe_to_excel(error_by_parent, os.path.join(OUT_DIR, "error_analysis_by_parent.xlsx"))
safe_to_excel(error_by_class, os.path.join(OUT_DIR, "error_analysis_by_class.xlsx"))

print("Error decomposition by fold:")
display(error_by_fold)

print("Error decomposition by parent:")
for model_name, sub in error_by_parent.groupby("model"):
    print("\nMODEL:", model_name)
    display(sub.pivot_table(index="true_L1", columns="error_type", values="pct_within_parent", fill_value=0))


In [ ]:
# =========================
# 14) Simple reporting figures
# =========================
def plot_metric_bar(df: pd.DataFrame, metric_col: str, title: str, out_path: str):
    if metric_col not in df.columns or len(df) == 0:
        print(f"Skipping plot; missing metric: {metric_col}")
        return

    plot_df = df.sort_values(metric_col, ascending=False).copy()

    plt.figure(figsize=(9, 5))
    plt.bar(plot_df["model"], plot_df[metric_col])
    plt.xticks(rotation=20, ha="right")
    plt.ylabel(metric_col)
    plt.title(title)
    plt.tight_layout()
    plt.savefig(out_path, dpi=220, bbox_inches="tight")
    plt.close()


plot_metric_bar(
    oof_summary_df,
    "oof_test_macro_f1",
    "Traditional baseline OOF test macro-F1",
    os.path.join(OUT_DIR, "baseline_oof_test_macro_f1.png"),
)

plot_metric_bar(
    oof_summary_df,
    "oof_test_HF1",
    "Traditional baseline OOF hierarchical F1",
    os.path.join(OUT_DIR, "baseline_oof_test_HF1.png"),
)

plot_metric_bar(
    cv_summary,
    "test_macro_f1_mean",
    "Traditional baseline test macro-F1 mean over folds",
    os.path.join(OUT_DIR, "baseline_cv_test_macro_f1_mean.png"),
)

print("Saved simple reporting figures in:", OUT_DIR)


## What to report in the paper

Use these files as the core traditional-baseline reporting artifacts:

1. `baseline_cv_oof_overall_summary.xlsx` — main held-out CV estimate using all out-of-fold test predictions.
2. `baseline_cv_summary_mean_std_over_folds.xlsx` — mean ± standard deviation across the five test folds.
3. `per_class_oof_report.xlsx` and `per_class_summary_mean_std_over_folds.xlsx` — L2 class-level analysis.
4. `per_parent_oof_report.xlsx` and `per_parent_summary_mean_std_over_folds.xlsx` — parent/L1 analysis.
5. `error_analysis_by_fold.csv`, `error_analysis_by_parent.csv`, and `error_analysis_by_class.csv` — hierarchical error decomposition.
6. `cm_oof_L2_*.png`, `cm_oof_parent_*.png`, `baseline_oof_test_macro_f1.png`, and `baseline_oof_test_HF1.png` — figures for the results section.